# Notebook 04 — Efficiency Report

Measures and visualizes efficiency across all 6 models:
- **Dashboard table**: Accuracy, Params, Train time/epoch, Inference latency, VRAM, Throughput
- **Plot 1**: Validation Accuracy vs Wall-clock Time (most important)
- **Plot 2**: Training Time per Epoch
- **Plot 3**: Accuracy vs Parameter Count (scatter + Pareto)
- **Plot 4**: Peak VRAM comparison
- **Plot 5**: Throughput comparison (nodes/sec, log scale)

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import json, csv as csv_mod
import torch
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from IPython.display import display

from reddit_gnn.config import (
    DEVICE, RESULTS_ROOT, CHECKPOINTS_DIR, NUM_CLASSES, NUM_FEATURES,
    DEFAULT_HPARAMS, SGC_DIR
)
from reddit_gnn.data.normalize import load_normalized_data
from reddit_gnn.training.utils import count_parameters
from reddit_gnn.analysis.efficiency import (
    measure_inference_latency, measure_gpu_memory, efficiency_dashboard
)

%matplotlib inline
plt.rcParams['figure.dpi'] = 120
sns.set_theme(style='whitegrid')

SAVE_DIR = os.path.join(RESULTS_ROOT, 'figures', '04_efficiency')
os.makedirs(SAVE_DIR, exist_ok=True)

MODEL_NAMES   = ['graphsage','graphsaint','sgc','gat','gatv2','cluster_gcn']
DISPLAY_NAMES = ['GraphSAGE','GraphSAINT','SGC','GAT','GATv2','ClusterGCN']

data, _, _ = load_normalized_data()
print('Data loaded.')

## 1. Build Model Instances & Measure Parameters

In [ ]:
def build_model(model_name):
    hp = DEFAULT_HPARAMS[model_name]
    if model_name == 'graphsage':
        from reddit_gnn.models.graphsage import GraphSAGE
        return GraphSAGE(NUM_FEATURES, hp['hidden'], NUM_CLASSES, aggregator=hp['aggregator']).to(DEVICE)
    elif model_name == 'graphsaint':
        from reddit_gnn.models.graphsaint import GraphSAINTNet
        return GraphSAINTNet(NUM_FEATURES, hp['hidden'], NUM_CLASSES).to(DEVICE)
    elif model_name == 'sgc':
        from reddit_gnn.models.sgc import SGC
        return SGC(NUM_FEATURES, NUM_CLASSES).to(DEVICE)
    elif model_name == 'gat':
        from reddit_gnn.models.gat import GAT
        return GAT(NUM_FEATURES, NUM_CLASSES, hp['hidden_per_head'], hp['heads']).to(DEVICE)
    elif model_name == 'gatv2':
        from reddit_gnn.models.gatv2 import GATv2
        return GATv2(NUM_FEATURES, NUM_CLASSES, hp['hidden_per_head'], hp['heads']).to(DEVICE)
    elif model_name == 'cluster_gcn':
        from reddit_gnn.models.cluster_gcn import ClusterGCN
        return ClusterGCN(NUM_FEATURES, hp['hidden'], NUM_CLASSES).to(DEVICE)

param_counts = {m: count_parameters(build_model(m)) for m in MODEL_NAMES}
pd.DataFrame({'Model': DISPLAY_NAMES, 'Parameters': [param_counts[m] for m in MODEL_NAMES]})

## 2. Load Baseline Metrics (Accuracy)

In [ ]:
def load_acc(model):
    p = os.path.join(RESULTS_ROOT, model, 'baseline', 'default', 'seed0', 'metrics.json')
    if os.path.exists(p):
        with open(p) as f: return json.load(f).get('test_acc', 0)
    return None

def load_history_for_timing(model, seed=0):
    p = os.path.join(RESULTS_ROOT, model, 'baseline', 'default', f'seed{seed}', 'history.csv')
    if not os.path.exists(p): return None
    with open(p) as f: return list(csv_mod.DictReader(f))

accs = {m: load_acc(m) for m in MODEL_NAMES}
print('Accuracies:', {k: f'{v:.4f}' if v else 'N/A' for k,v in accs.items()})

## 3. Measure Inference Latency & VRAM

In [ ]:
latencies = {}
vrams     = {}

for model_name in MODEL_NAMES:
    print(f'Measuring {model_name}...')
    model = build_model(model_name)
    
    try:
        if model_name == 'sgc':
            K = DEFAULT_HPARAMS['sgc'].get('K', 2)
            X_K = torch.load(os.path.join(SGC_DIR, f'reddit_sgc_K{K}.pt'), weights_only=False)
            import copy; sgc_data = copy.copy(data); sgc_data.x = X_K
            ms, _ = measure_inference_latency(model, sgc_data.cpu(), DEVICE, model_type='sgc', n_runs=3)
            vrams[model_name] = 0.0  # SGC: negligible GPU usage
        else:
            ms, _ = measure_inference_latency(model, data, DEVICE, n_runs=3)
            vrams[model_name] = measure_gpu_memory(model, data, DEVICE)
        latencies[model_name] = ms
        print(f'  Latency: {ms:.1f}ms, VRAM: {vrams.get(model_name,0):.0f}MB')
    except Exception as e:
        print(f'  Failed: {e}')
        latencies[model_name] = 0.0
        vrams[model_name] = 0.0

## 4. Efficiency Dashboard Table

In [ ]:
# Compute epoch times from training history
epoch_times = {}
for m in MODEL_NAMES:
    hist = load_history_for_timing(m)
    if hist and 'epoch_time_s' in hist[0]:
        times = [float(h['epoch_time_s']) for h in hist]
        epoch_times[m] = np.mean(times)
    else:
        epoch_times[m] = None

rows = []
for model, display in zip(MODEL_NAMES, DISPLAY_NAMES):
    rows.append({
        'Model': display,
        'Test Acc': f"{accs.get(model, 'N/A'):.4f}" if accs.get(model) else 'N/A',
        'Params': f"{param_counts[model]:,}",
        'Epoch Time (s)': f"{epoch_times.get(model, 'N/A'):.1f}" if epoch_times.get(model) else 'N/A',
        'Inference (ms)': f"{latencies.get(model, 0):.1f}",
        'VRAM (MB)': f"{vrams.get(model, 0):.0f}",
    })

df_eff = pd.DataFrame(rows)
display(df_eff.style.set_caption('Efficiency Dashboard — All 6 Models').hide(axis='index'))

eff_path = os.path.join(SAVE_DIR, 'efficiency_metrics.json')
with open(eff_path, 'w') as f:
    json.dump(rows, f, indent=2)
print(f'Saved: {eff_path}')

## 5. Plot 1 — Validation Accuracy vs Wall-Clock Time

In [ ]:
fig, ax = plt.subplots(figsize=(13, 7))
colors = sns.color_palette('husl', len(MODEL_NAMES))

for model, display, color in zip(MODEL_NAMES, DISPLAY_NAMES, colors):
    hist = load_history_for_timing(model)
    if hist is None or 'epoch_time_s' not in hist[0]: continue
    
    cum_time = np.cumsum([float(h['epoch_time_s']) for h in hist]) / 60  # minutes
    val_acc  = [float(h['val_acc']) for h in hist]
    ax.plot(cum_time, val_acc, label=display, color=color, lw=2)

ax.set_xlabel('Wall-Clock Time (minutes)', fontsize=13)
ax.set_ylabel('Validation Accuracy', fontsize=13)
ax.set_title('Validation Accuracy vs Training Time — All 6 Models', fontsize=14)
ax.legend(fontsize=11); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'acc_vs_time.png'), dpi=150)
plt.show()

## 6. Plot 3 — Accuracy vs Parameter Count (Scatter + Pareto)

In [ ]:
fig, ax = plt.subplots(figsize=(11, 7))
colors = sns.color_palette('husl', len(MODEL_NAMES))
xs, ys = [], []

for model, display, color in zip(MODEL_NAMES, DISPLAY_NAMES, colors):
    acc = accs.get(model)
    if acc is None: continue
    p = param_counts[model]
    xs.append(p); ys.append(acc)
    ax.scatter(p, acc, s=200, color=color, label=display, zorder=5, edgecolors='black', lw=0.8)
    ax.annotate(display, (p, acc), textcoords='offset points', xytext=(8, 4), fontsize=10)

# Pareto frontier
if xs:
    sorted_pts = sorted(zip(xs, ys))
    pareto_x, pareto_y, max_y = [], [], -1
    for px, py in sorted_pts:
        if py > max_y:
            pareto_x.append(px); pareto_y.append(py); max_y = py
    if len(pareto_x) > 1:
        ax.plot(pareto_x, pareto_y, 'k--', lw=1.5, alpha=0.5, label='Pareto frontier')

ax.set_xlabel('Trainable Parameters', fontsize=12)
ax.set_ylabel('Test Accuracy', fontsize=12)
ax.set_title('Accuracy vs Model Complexity — Pareto Analysis', fontsize=13)
ax.legend(fontsize=10); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'accuracy_vs_params.png'), dpi=150)
plt.show()

## 7. Plot 5 — Throughput & VRAM Comparison

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(20, 6))
colors = sns.color_palette('husl', len(MODEL_NAMES))

# Inference latency
lat_vals = [latencies.get(m, 0) for m in MODEL_NAMES]
axes[0].barh(DISPLAY_NAMES, lat_vals, color=colors, edgecolor='black', lw=0.5)
axes[0].set_xlabel('Inference Latency (ms)')
axes[0].set_title('Inference Latency (full graph)')
for i, v in enumerate(lat_vals):
    if v > 0: axes[0].text(v*1.01, i, f'{v:.0f}ms', va='center', fontsize=9)

# VRAM
vram_vals = [vrams.get(m, 0) for m in MODEL_NAMES]
axes[1].barh(DISPLAY_NAMES, vram_vals, color=colors, edgecolor='black', lw=0.5)
axes[1].set_xlabel('Peak VRAM (MB)')
axes[1].set_title('Peak GPU Memory (Forward+Backward)')
for i, v in enumerate(vram_vals):
    if v > 0: axes[1].text(v*1.01, i, f'{v:.0f}MB', va='center', fontsize=9)

# Parameters
param_vals = [param_counts[m]/1000 for m in MODEL_NAMES]
axes[2].barh(DISPLAY_NAMES, param_vals, color=colors, edgecolor='black', lw=0.5)
axes[2].set_xlabel('Parameters (K)')
axes[2].set_title('Trainable Parameters')
for i, v in enumerate(param_vals):
    axes[2].text(v*1.01, i, f'{v:.0f}K', va='center', fontsize=9)

plt.suptitle('Efficiency Comparison — All 6 Models', fontsize=14)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'efficiency_trio.png'), dpi=150)
plt.show()

print(f'\nAll efficiency figures saved to: {SAVE_DIR}')